# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [61]:
# Only needed for Udacity workspace

import importlib.util
import sys
import chromadb
# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [62]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
from dotenv import load_dotenv
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from typing import List, Any, Annotated
from pydantic import BaseModel, Field
import json
from lib.parsers import PydanticOutputParser, JsonOutputParser

In [63]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

load_dotenv()
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY= os.getenv("OPENAI_BASE_URL")
api_key = os.getenv("TAVILY_API_KEY")
chat_model = LLM()

In [64]:
# class GameSummary(BaseModel):
#     """Represents a single action item from a meeting"""
#     Platform: Annotated[str, Field(description="Game Platform")]
#     Name: Annotated[str, Field(description="Name of the game.")]
#     YearOfRelease: Annotated[str, Field(description="When the game was released")]
#     Description: Annotated[str, Field(description="Description of the game")]


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [65]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

from lib.tooling import tool
import chromadb
import sqlite3

@tool
def retrieve_game(query: str) -> list[str]:
    """Retrieve relevant games from the vector database based on a semantic query.
    Args:
        query: The search query to match against games.
    """
    # Query the collection. ChromaDB automatically generates the embedding for the query.
    chroma_client = chromadb.PersistentClient(path="./games/")
    collection = chroma_client.get_collection("udaplayts")
    results = collection.query(
        query_texts=[query],
        n_results=2  # Returns top 2 most similar games
    )
    
    # Chroma returns results in the format: {'documents': [[doc1, doc2]]}
    # We extract and return the inner list of documents.
    return results.get("documents", [[]])[0]


In [66]:
retrieved_docs = retrieve_game(query="Spider-Man")
print (retrieved_docs)

["[PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.", "[PlayStation 5] Marvel's Spider-Man 2 (2023) - The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters."]


#### Evaluate Retrieval Tool

In [67]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [68]:
from lib.llm import LLM
from lib.parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the structured output schema
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")

#Implement the tool with proper arguments, type annotations, and docstring
@tool
def evaluate_retrieval(question: str, retrieved_docs:list[str]) -> str:
        """Assess the quality of the retrieved game documents to see if they can answer the user's question.
        
        Args:
            question: The user's original question.
            retrieved_docs: List of retrieved documents from the vector db
            {retrieved_docs}
        """
        llm = LLM(model="gpt-4o-mini") #instantiate  llm
        docs_formatted = "\n".join([f"- {doc}" for doc in retrieved_docs])
        # Craft the prompt for the LLM judge
        prompt = (
            "Your task is to evaluate if the documents are enough to respond to the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
            f"Query: {question}\n\n"
            f"Retrieved Documents:\n{docs_formatted}"
        )

    # Invoke the LLM with the Pydantic schema for structured output
        response = llm.invoke(input=prompt,
        response_format=EvaluationReport)

        parser = PydanticOutputParser(model_class=EvaluationReport)
        return parser.parse(response)

In [69]:
# Step 1: define the user's question
user_question = "When was Pokémon Gold and Silver released?"
# Step 2: retrieve relevant docs from the vector DB
retrieved_docs = retrieve_game(query=user_question)
print("Retrieved docs:", retrieved_docs)

# Step 3: pass BOTH the question AND the retrieved docs to evaluate_retrieval
evaluation = evaluate_retrieval(question=user_question, retrieved_docs=retrieved_docs)
print(f"\nUseful:      {evaluation.useful}")
print(f"Description: {evaluation.description}")

Retrieved docs: ['[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.']

Useful:      False
Description: The retrieved documents do not provide a direct answer to the query regarding the release date of Pokémon Gold and Silver. While the first document mentions that Pokémon Gold and Silver were released in 1999, it does not specify the exact date or month of the release. The second document is irrelevant as it discusses Pokémon Ruby and Sapphire, which are not related to the query. Therefore, the information is insufficient to fully answer the question about the release date of Pokémon Gold and Silver.


#### Game Web Search Tool

In [70]:
from dotenv import parser
from datetime import datetime
from tavily import TavilyClient
import os
from dotenv import load_dotenv
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(query: str, search_depth: str = "advanced"):
    """
    Search the web using Tavily API
    args:
        - question: a question about game industry.
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    load_dotenv()
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

In [71]:
# Step 1: define the user's question
user_question = "When was Pokémon Gold and Silver released?"
# Step 2: retrieve relevant docs from the vector DB
retrieved_docs = retrieve_game(query=user_question)
evaluation = evaluate_retrieval(question=user_question, retrieved_docs=retrieved_docs)
print("Retrieved docs:", retrieved_docs)

parser = PydanticOutputParser(model_class=EvaluationReport)
eval_useful = evaluation.useful
print (eval_useful)

if not eval_useful:
    web_results=game_web_search(query=user_question)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    
    print(web_results)
else:
    print("Retrieval was sufficient. No web search needed.")

Retrieved docs: ['[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.']
False
{'answer': 'Pokémon Gold and Silver were released in Japan on November 21, 1999, in North America on October 15, 2000, and in Europe on April 6, 2001.', 'results': [{'url': 'https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver', 'title': 'Pokémon Gold and Silver - Wikipedia', 'content': 'In September 1999, Nintendo announced that Gold and Silver would be released in North America in September 2000. In May 2000, Nintendo announced the official North American release date of Gold and Silver would instead be October 16 of that year. The release date was later changed to October 15. In North America, Nintendo started accepting pre-orders for the games in August; a CD-R

### Agent

In [72]:
from lib.state_machine import Run
from typing import TypedDict,List, Optional
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
from lib.tooling import Tool, ToolCall, tool
from lib.state_machine import (
    StateMachine,
    Step,
    EntryPoint,
    Termination,
)

#Schema of shared state
class AgentState(TypedDict):
    user_query: str  # The current user query being processed
    instructions: str  # System instructions for the agent
    messages: List[dict]  # List of conversation messages
    retrieved_docs:List[str]
    evaluation_useful:bool
    evaluation_desc:str
    web_results: Optional[dict]
    messages:List[dict]
    final_answer:str

#agent class
class Agent:
    def __init__(
        self,
        role:str="Personal Assistant",
        model_name:str="gpt-4o-mini",
        instructions:str="",

    ):
        self.role:role
        self.model_name=model_name
        self.instructions =instructions
        # Initialize state machine
        self.workflow = self._create_state_machine()
      
#Call all functions here with params
    def ret_step(self, state:AgentState)->AgentState :
        query = state["user_query"]
        docs = retrieve_game(query=query)
        return {
            "retrieved_docs":docs
        }
    def eva_step (self, state:AgentState)->AgentState:
        question=state["user_query"]
        retrieved_docs = state["retrieved_docs"]
        evaluation = evaluate_retrieval(question, retrieved_docs)
        return {
            "evaluation_useful":evaluation.useful,
            "evaluation_desc":evaluation.description
        }

    def web_search (self, state:AgentState)->AgentState:
        query=state["user_query"]
        web_results=game_web_search(query)

        return {
            "web_results":web_results
        }
    def answer(self, state:AgentState)->AgentState:
        llm = LLM(model=self.model_name)

        #FinalStringLLM
        StringLLM=""
        if state["retrieved_docs"]:
            StringLLM +="Retrieved Docs".join(state["retrieved_docs"])
            if state.get("web_results"):
                StringLLM +=f"\nWeb Results:\n{json.dumps(state['web_results'], indent=2)}"

        prompt = (
            f"you are udaplay agent, an expert in video game industry. \n"
            f"System Instructions: {state["instructions"]}\n"
            f"StringLLM: {StringLLM}\n"
            f"User Query: {state["user_query"]}\n"
           
        )

        response = llm.invoke(prompt)
        return {
            "final_answer":response.content
            
        }
#-----------------------
    def _create_state_machine(self)->StateMachine[AgentState]:
            machine = StateMachine[AgentState](AgentState)

            #Create Steps
            entry = EntryPoint[AgentState]()
            retrieval = Step[AgentState]("retrieved_docs", self.ret_step)
            evaluation=Step[AgentState]("EvaluationReport",self.eva_step)
            websearch = Step[AgentState]("web_results",self.web_search)
            answer = Step[AgentState]("Answer",self.answer)
            termination = Termination[AgentState]()
            #Add Steps to workflow
            machine.add_steps([entry, retrieval, evaluation, websearch,answer, termination])
            #Connect
            machine.connect(entry,retrieval)
            machine.connect(retrieval,evaluation)

            #from evaluation to web search is conditional

            def check_eval(state:AgentState)->AgentState:
                if not state.get("evaluation_useful", True):
                    return websearch
                return answer

            machine.connect(evaluation,[websearch,answer],check_eval)
            machine.connect(websearch,answer)
            machine.connect(answer,termination)

            return machine
#---------------------------
    def invoke(self,query:str)->Run:
        """Run the state machine"""
        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "retrieved_docs": [],
            "evaluation_useful": True,
            "evaluation_desc": "",
            "web_results": None,
            "final_answer": "",
            "messages": []
        }

        run_object = self.workflow.run(initial_state)

        return run_object

In [73]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
#tools = [game_web_search, retrieved_docs,evaluate_retrieval]
GameAgent = Agent(
    model_name="gpt-4o-mini",
    instructions=( "Provide clear, accurate historical information about game releases and platforms. "
    "If information is derived from the web, mention that it was confirmed via web search.")
)


In [74]:
run_object = GameAgent.invoke(
    query="When Pokémon Gold and Silver was released?"
)

run_object.get_final_state()["final_answer"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieved_docs
[StateMachine] Executing step: EvaluationReport
[StateMachine] Executing step: web_results
[StateMachine] Executing step: Answer
[StateMachine] Terminating: __termination__


'Pokémon Gold and Silver were released in Japan on November 21, 1999. The games were later released in North America on October 15, 2000, and in Europe on April 6, 2001. These titles marked the beginning of the second generation of Pokémon games and were developed for the Game Boy Color.'

In [75]:
run_object = GameAgent.invoke(
    query="Which one was the first 3D platformer Mario game?"
)

run_object.get_final_state()["final_answer"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieved_docs
[StateMachine] Executing step: EvaluationReport
[StateMachine] Executing step: Answer
[StateMachine] Terminating: __termination__


'The first 3D platformer featuring Mario was **Super Mario 64**, released in 1996 for the Nintendo 64. This game was groundbreaking for its time, introducing players to a fully 3D environment and setting new standards for the platforming genre. In **Super Mario 64**, players control Mario as he embarks on a quest to rescue Princess Peach from Bowser, navigating through various worlds and completing challenges.'

In [76]:
run_object = GameAgent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?"
)

run_object.get_final_state()["final_answer"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieved_docs
[StateMachine] Executing step: EvaluationReport
[StateMachine] Executing step: web_results
[StateMachine] Executing step: Answer
[StateMachine] Terminating: __termination__


'Mortal Kombat X was not originally released for the PlayStation 5; it was launched on April 14, 2015, for PlayStation 4, Xbox One, and PC. However, it is available to play on the PlayStation 5 through the PlayStation Plus Collection, which allows PS5 users to access a selection of PS4 games, including Mortal Kombat X. This information has been confirmed via web search.'

### (Optional) Advanced

In [77]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes